Loading the Libraries


In [1]:
import pandas as pd
import re
import os


file paths

In [ ]:

# # -----------------------------
# # CONFIG
# # -----------------------------
# CSV_PATH = "/Users/vaidehipandya/Desktop/Work/loaded_data/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv"
# TARGET_SLIDE = "TCGA-55-7574-01Z-00-DX1"
# TILE_SIZE = 224   # adjust if your tiles are e.g. 256 or 400 µm → 222 px


# # -----------------------------
# # LOAD CSV
# # -----------------------------
# df = pd.read_csv(CSV_PATH)

# print("Loaded:", df.shape, "rows")

# # -----------------------------
# # FILTER FOR TARGET SLIDE
# # -----------------------------
# df_slide = df[df["slides"] == TARGET_SLIDE].copy()
# print("Subset rows:", df_slide.shape)

# # -----------------------------
# # REMOVE LEIDEN CLUSTER COLUMNS
# # -----------------------------
# leiden_cols = [c for c in df_slide.columns if "leiden" in c.lower()]
# df_slide = df_slide.drop(columns=leiden_cols)
# print("Removed leiden columns:", leiden_cols)

#  #REMOVE HPC ID COLUMN
# # -----------------------------
# if "hpc_id" in df_slide.columns:
#     df_slide = df_slide.drop(columns=["hpc_id"])
#     print("Removed: hpc_id")
# else:
#     print("hpc_id not found")


# print(df_slide.head())
# print(df_slide.columns)


import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH = "/Users/vaidehipandya/Desktop/Work/loaded_data/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv"
OUT_CSV = "tile_coordinates_load_TCGA_LUAD_5x.csv"

DATASET_ID = "TCGA_LUAD_5x"
TILE_SIZE_5X = 224

# -----------------------------
# LOAD CSV
# -----------------------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape, "rows")

# -----------------------------
# KEEP ONLY REQUIRED COLUMNS
# -----------------------------
df = df[["slides", "tiles"]].copy()

# -----------------------------
# PARSE TILE COORDS
# -----------------------------
def parse_tile(tile_name: str):
    base = str(tile_name).replace(".jpeg", "").replace(".jpg", "")
    col_str, row_str = base.split("_")
    return int(col_str), int(row_str)

df["col"], df["row"] = zip(*df["tiles"].apply(parse_tile))

# -----------------------------
# ADD GEOMETRY COLUMNS
# -----------------------------
df["x_5x"] = df["col"] * TILE_SIZE_5X
df["y_5x"] = df["row"] * TILE_SIZE_5X

df["dataset_id"] = DATASET_ID

df["slide_tile"] = (
    df["slides"].astype(str).str.strip()
    + "_"
    + df["tiles"].astype(str).str.strip()
)

# -----------------------------
# FINAL COLUMN ORDER
# -----------------------------
final_cols = [
    "dataset_id",
    "slides",
    "tiles",
    "slide_tile",
    "col",
    "row",
    "x_5x",
    "y_5x"
]

df[final_cols].to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print("Unique slides:", df["slides"].nunique())
print(df.head())




Loaded: (360567, 5) rows
Subset rows: (254, 5)
Removed leiden columns: []
Removed: hpc_id
        samples                   slides       tiles  \
0  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  18_15.jpeg   
1  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   14_6.jpeg   
2  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  11_23.jpeg   
3  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1  10_20.jpeg   
4  TCGA-55-7574  TCGA-55-7574-01Z-00-DX1   23_1.jpeg   

                           slide_tile  
0  TCGA-55-7574-01Z-00-DX1_18_15.jpeg  
1   TCGA-55-7574-01Z-00-DX1_14_6.jpeg  
2  TCGA-55-7574-01Z-00-DX1_11_23.jpeg  
3  TCGA-55-7574-01Z-00-DX1_10_20.jpeg  
4   TCGA-55-7574-01Z-00-DX1_23_1.jpeg  
Index(['samples', 'slides', 'tiles', 'slide_tile'], dtype='object')


In [ ]:
import openslide

# Replace with your actual WSI path
SLIDE_PATH = "/Users/vaidehipandya/Desktop/Work/Heatmap/TCGA-55-7574-01Z-00-DX1.09639e6a-d85f-4d84-abbc-0f5a6d679683.svs"
5
slide = openslide.OpenSlide(SLIDE_PATH)

print("Levels:", slide.level_count)
print("Level dimensions:", slide.level_dimensions)
print("Downsamples:", slide.level_downsamples)

# Microns-per-pixel
mpp_x = float(slide.properties.get("openslide.mpp-x", "0"))
mpp_y = float(slide.properties.get("openslide.mpp-y", "0"))
print("Native MPP:", mpp_x, mpp_y)

Levels: 4
Level dimensions: ((45084, 54285), (11271, 13571), (2817, 3392), (1408, 1696))
Downsamples: (1.0, 4.0000368432687345, 16.004046199037514, 32.013775728987994)
Native MPP: 0.252 0.252


In [5]:
# --- Step 1: Understand WSI native resolution vs tiling resolution ---

native_mpp = 0.252     # microns per pixel at level 0 (from OpenSlide)
target_mpp = 1.8       # microns per pixel used for tiling in the paper

scale = target_mpp / native_mpp

print("Native MPP:", native_mpp)
print("Target MPP:", target_mpp)
print("Scaling factor (1.8µm-px grid → native pixels):", scale)

# --- Step 2: Decide tile size in microns (from paper spec) ---

tile_size_um = 400   # tile diameter in microns, as described in the paper

tile_size_px = tile_size_um / target_mpp

print("Tile size in pixels at 1.8µm/px:", tile_size_px)

Native MPP: 0.252
Target MPP: 1.8
Scaling factor (1.8µm-px grid → native pixels): 7.142857142857143
Tile size in pixels at 1.8µm/px: 222.22222222222223


In [6]:
# --- Step 4: Extract row and column indices from tile filenames ---

def parse_tile(tile_name):
    # remove extension (.jpeg or .jpg)
    base = tile_name.replace(".jpeg", "").replace(".jpg", "")
    col_str, row_str = base.split("_")
    return int(col_str), int(row_str)

# Apply to your dataframe
df_slide["col"], df_slide["row"] = zip(*df_slide["tiles"].apply(parse_tile))

print(df_slide[["slides","tiles", "col", "row"]].head())

                    slides       tiles  col  row
0  TCGA-55-7574-01Z-00-DX1  18_15.jpeg   18   15
1  TCGA-55-7574-01Z-00-DX1   14_6.jpeg   14    6
2  TCGA-55-7574-01Z-00-DX1  11_23.jpeg   11   23
3  TCGA-55-7574-01Z-00-DX1  10_20.jpeg   10   20
4  TCGA-55-7574-01Z-00-DX1   23_1.jpeg   23    1


In [7]:
# --- Step 5: Compute pixel coordinates in the 1.8 µm grid (224 px tiles) ---

TILE_SIZE_5X = 224  # actual tile size used for training

df_slide["x_5x"] = df_slide["col"] * TILE_SIZE_5X
df_slide["y_5x"] = df_slide["row"] * TILE_SIZE_5X

df_slide[["slides","tiles", "col", "row", "x_5x", "y_5x"]].head()

,slides,tiles,col,row,x_5x,y_5x
0,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,18,15,4032,3360
1,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,14,6,3136,1344
2,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,11,23,2464,5152
3,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,10,20,2240,4480
4,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,23,1,5152,224


In [8]:
# --- Step 6: Convert tile coordinates to native WSI resolution ---

native_mpp = 0.252
target_mpp = 1.8

scale = target_mpp / native_mpp    # ≈ 7.142857
TILE_SIZE_5X = 224
TILE_SIZE_NATIVE = int(TILE_SIZE_5X * scale)   # ≈ 1600

print("Scale factor:", scale)
print("Tile size native (px):", TILE_SIZE_NATIVE)

# Convert 1.8µm grid → native pixel coordinates
df_slide["x_native"] = df_slide["col"] * TILE_SIZE_NATIVE
df_slide["y_native"] = df_slide["row"] * TILE_SIZE_NATIVE

df_slide[["slides","tiles", "col", "row", "x_native", "y_native"]].head()

Scale factor: 7.142857142857143
Tile size native (px): 1600


,slides,tiles,col,row,x_native,y_native
0,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,18,15,28800,24000
1,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,14,6,22400,9600
2,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,11,23,17600,36800
3,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,10,20,16000,32000
4,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,23,1,36800,1600


In [ ]:
import h5py

H5_PATH = "/Users/vaidehipandya/Desktop/Work/subset_all_10000_images.h5"

with h5py.File(H5_PATH, "r") as f:
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, obj.shape, obj.dtype)
    f.visititems(visitor)

In [20]:
import h5py



with h5py.File(H5_PATH, "r") as f:
    print("Top level keys:", list(f.keys()))
    print("\nAll datasets:")
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            shape = obj.shape
            dtype = obj.dtype
            print(f"  {name}  shape={shape}  dtype={dtype}")
    f.visititems(visitor)

Top level keys: ['train_hist_subtype', 'train_img', 'train_labels', 'train_patterns', 'train_samples', 'train_slides', 'train_tiles']

All datasets:
  train_hist_subtype  shape=(10000,)  dtype=|S37
  train_img  shape=(10000, 224, 224, 3)  dtype=uint8
  train_labels  shape=(10000,)  dtype=float32
  train_patterns  shape=(10000,)  dtype=|S37
  train_samples  shape=(10000,)  dtype=|S12
  train_slides  shape=(10000,)  dtype=|S23
  train_tiles  shape=(10000,)  dtype=|S38


In [26]:
import numpy as np
import h5py

# df_slide must have a column "tiles"
# H5_PATH must point to your .h5 file

with h5py.File(H5_PATH, "r") as f:
    h5_tiles = f["train_tiles"][:]  # bytes, length 10000

# decode bytes -> str
h5_tiles = [t.decode("utf-8") for t in h5_tiles]

# tile_id -> global index in train_img
tile_to_idx = {tile_id: i for i, tile_id in enumerate(h5_tiles)}

# map csv tile ids to h5 indices
df_slide = df_slide.copy()
df_slide["h5_index"] = df_slide["tiles"].map(tile_to_idx)

# check
missing = df_slide["h5_index"].isna().sum()
print("Missing h5_index:", missing)

# if missing > 0, show examples
if missing > 0:
    print("Example missing tile IDs:", df_slide.loc[df_slide["h5_index"].isna(), "tiles"].head(10).tolist())

Missing h5_index: 0


In [28]:
# --- Save final coordinates for Streamlit viewer ---

OUTPUT_CSV = "TCGA_55_7574_final_tile_coordinates.csv"

df_slide["slide_tile"] = (
    df_slide["slides"].astype(str).str.strip().str.upper()
    + "_"
    + df_slide["tiles"].astype(str).str.strip()
)


final_cols = [
    "slides",
    "tiles",
    "slide_tile",
    "col", "row",
    "x_5x", "y_5x",
    "x_native", "y_native",
    "h5_index" 
]

df_slide[final_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved Streamlit coordinate file →", OUTPUT_CSV)
df_slide[final_cols].head(20)

Saved Streamlit coordinate file → TCGA_55_7574_final_tile_coordinates.csv


,slides,tiles,slide_tile,col,row,x_5x,y_5x,x_native,y_native,h5_index
0,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,18,15,4032,3360,28800,24000,8981
1,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,14,6,3136,1344,22400,9600,8982
2,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,11,23,2464,5152,17600,36800,8988
3,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,10,20,2240,4480,16000,32000,8820
4,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,23,1,5152,224,36800,1600,1473
5,TCGA-55-7574-01Z-00-DX1,11_24.jpeg,TCGA-55-7574-01Z-00-DX1_11_24.jpeg,11,24,2464,5376,17600,38400,8992
6,TCGA-55-7574-01Z-00-DX1,4_23.jpeg,TCGA-55-7574-01Z-00-DX1_4_23.jpeg,4,23,896,5152,6400,36800,4617
7,TCGA-55-7574-01Z-00-DX1,10_12.jpeg,TCGA-55-7574-01Z-00-DX1_10_12.jpeg,10,12,2240,2688,16000,19200,9003
8,TCGA-55-7574-01Z-00-DX1,11_11.jpeg,TCGA-55-7574-01Z-00-DX1_11_11.jpeg,11,11,2464,2464,17600,17600,9019
9,TCGA-55-7574-01Z-00-DX1,4_20.jpeg,TCGA-55-7574-01Z-00-DX1_4_20.jpeg,4,20,896,4480,6400,32000,6429


In [22]:
df = df_slide[df_slide["slides"] == "TCGA-55-7574-01Z-00-DX1"]

print("Unique tile widths:", (df["x_native"].sort_values().diff().dropna().unique()))
print("Unique tile heights:", (df["y_native"].sort_values().diff().dropna().unique()))

Unique tile widths: [   0. 1600.]
Unique tile heights: [1600.    0.]


In [23]:
dupes = df[df.duplicated(subset=["x_native", "y_native"], keep=False)]
dupes.head(20)

,samples,slides,tiles,slide_tile,col,row,x_5x,y_5x,x_native,y_native


In [7]:

df_slide = pd.read_csv("/Users/vaidehipandya/Desktop/Work/Heatmap/TCGA_55_7574_final_tile_coordinates.csv")
df_slide = df_slide.copy()

df_slide["slide_tile"] = (
    df_slide["slides"].astype(str).str.strip()
    + "_"
    + df_slide["tiles"].astype(str).str.strip()
)

df_slide.head(20)

final_cols = [
    "slide_tile",
    "slides",
    "tiles",
    "col", "row",
    "x_5x", "y_5x",
    "x_native", "y_native"
]

# Save file
OUTPUT_CSV = "TCGA_55_7574_final_tile_coordinates.csv"
df_slide[final_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved Streamlit coordinate file →", OUTPUT_CSV)
df_slide[final_cols].head(10)

Saved Streamlit coordinate file → TCGA_55_7574_final_tile_coordinates.csv


,slide_tile,slides,tiles,col,row,x_5x,y_5x,x_native,y_native
0,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,18,15,4032,3360,28800,24000
1,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,14,6,3136,1344,22400,9600
2,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,11,23,2464,5152,17600,36800
3,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,10,20,2240,4480,16000,32000
4,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,23,1,5152,224,36800,1600
5,TCGA-55-7574-01Z-00-DX1_11_24.jpeg,TCGA-55-7574-01Z-00-DX1,11_24.jpeg,11,24,2464,5376,17600,38400
6,TCGA-55-7574-01Z-00-DX1_4_23.jpeg,TCGA-55-7574-01Z-00-DX1,4_23.jpeg,4,23,896,5152,6400,36800
7,TCGA-55-7574-01Z-00-DX1_10_12.jpeg,TCGA-55-7574-01Z-00-DX1,10_12.jpeg,10,12,2240,2688,16000,19200
8,TCGA-55-7574-01Z-00-DX1_11_11.jpeg,TCGA-55-7574-01Z-00-DX1,11_11.jpeg,11,11,2464,2464,17600,17600
9,TCGA-55-7574-01Z-00-DX1_4_20.jpeg,TCGA-55-7574-01Z-00-DX1,4_20.jpeg,4,20,896,4480,6400,32000
